In [2]:
from datetime import time
from os import times
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as patches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import math
from cartopy.mpl.ticker import LongitudeFormatter,LatitudeFormatter

#math的相关函数，import后可以直接使用
from math import radians
from math import sin
from math import cos
from math import asin
from math import sqrt

# np.set_printoptions(threshold=np.inf)#显示所有的数组
from geopy.distance import great_circle
import matplotlib
# matplotlib.rcParams['font.family'] = 'Times New Roman'
import seaborn as sns
from scipy import stats

# from jupyterthemes import jtplot
# jtplot.style()


In [3]:
infile = xr.open_dataset('E:/google/IBTrACS.WP.v04r01.nc')
infile

<xarray.Dataset> Size: 1GB
Dimensions:           (storm: 4176, date_time: 360, quadrant: 4)
Coordinates:
    time              (storm, date_time) datetime64[ns] 12MB ...
    lat               (storm, date_time) float32 6MB ...
    lon               (storm, date_time) float32 6MB ...
Dimensions without coordinates: storm, date_time, quadrant
Data variables: (12/159)
    numobs            (storm) float32 17kB ...
    sid               (storm) |S13 54kB ...
    season            (storm) float32 17kB ...
    number            (storm) int16 8kB ...
    basin             (storm, date_time) |S2 3MB ...
    subbasin          (storm, date_time) |S2 3MB ...
    ...                ...
    reunion_gust      (storm, date_time) float32 6MB ...
    reunion_gust_per  (storm, date_time) float32 6MB ...
    usa_seahgt        (storm, date_time) float32 6MB ...
    usa_searad        (storm, date_time, quadrant) float32 24MB ...
    storm_speed       (storm, date_time) float32 6MB ...
    storm_dir         (storm, date_time) float32 6MB ...
Attributes: (12/49)
    title:                      IBTrACS - International Best Track Archive fo...
    summary:                    The intent of the IBTrACS project is to overc...
    source:                     The original data are tropical cyclone positi...
    Conventions:                ACDD-1.3
    Conventions_note:           Data are nearly CF-1.7 compliant. The sole is...
    product_version:            v04r01
    ...                         ...
    history:                    Tue Sep 24 05:54:34 2024: ncks --no_abc --cnk...
    license:                    These data may be redistributed and used with...
    featureType:                trajectory
    cdm_data_type:              Trajectory
    comment:                    The tracks of TCs generally look like a traje...
    NCO:                        netCDF Operators version 5.0.7 (Homepage = ht...

In [4]:
isotime=infile['iso_time'].values

In [5]:
df1 = pd.read_excel(f'Typhoon_Cluster_3.xlsx')
indallc1=np.array(df1['Typhoon ID'])
indallc1.shape

(413,)

413个台风

In [6]:
timec1=isotime[indallc1]
timec1_datetime = np.array([np.datetime64(time.decode('utf-8')) for time in timec1.flatten() if time])

35916个时间，但是存在交集，某些时间可能会重合，所以实际上的时间比这个少

In [27]:

for i in range(1982,2024):
#     print(i)
    data= xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt=data['z']/9.80665
    res_time = data['valid_time']
    
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    hgt500=hgt.sel(valid_time=common_times)#每年的hgt
    hgtwp = hgt500.loc[:,500,60:10,90:180]
    mewp=hgtwp.mean('valid_time')#一年的平均位势高度
    lon=hgtwp['longitude']

    for j in range(mewp.shape[1]):
        if mewp[:,j].max()>=5880:            
            wpsh_wp[i-1982] = lon[j]
            break
    if wpsh_wp[i-1982]==0:
        for j in range(mewp.shape[0]):
            if mewp[:,j].max()>=5870:
                wpsh_wp[i-1982] = lon[j]
                break    
    if wpsh_wp[i-1982]==0:
        for j in range(mewp.shape[0]):
            if mewp[:,j].max()>=5860:
                wpsh_wp[i-1982] = lon[j]
                break   
    if wpsh_wp[i-1982]==0:
        for j in range(mewp.shape[0]):
            if mewp[:,j].max()>=5850:
                wpsh_wp[i-1982] = lon[j]
                break   
print(wpsh_wp)


[155. 140. 125. 140. 165. 145. 155. 155. 150. 145. 170. 140. 140. 145.
 155. 145. 140. 140. 150. 160. 150. 145. 140. 140. 155. 155. 155. 145.
 135. 150. 150. 145. 140. 150. 140. 120. 140. 135. 135. 140. 110. 115.]


In [31]:
np.max(mewp)

<xarray.DataArray 'z' ()> Size: 8B
array(5905.52978516)
Coordinates:
    number          int64 8B ...
    pressure_level  float64 8B 500.0

In [37]:
result=[]
for i in range(1982,2024):
#     print(i)
    data= xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt=data['z']/9.80665
    res_time = data['valid_time']
    
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    hgt500=hgt.sel(valid_time=common_times)#每年的hgt
    hgtwp = hgt500.loc[:,500,60:10,110:180]
    mewp=hgtwp.mean('valid_time')#一年的平均位势高度
    lon=hgtwp['longitude']
    resum=0
    for j in range(11):
        for k in range(15):
            if 5890>mewp[j,k]>=5880:
                resum=resum+1
            elif 5900>mewp[j,k]>=5890:
                resum=resum+2
            elif 5910>mewp[j,k]>=5900:
                resum=resum+3
            elif 5920>mewp[j,k]>=5910:
                resum=resum+4
    print(resum)
    result.append(resum)




2
23
0
0
2
20
14
7
21
26
2
25
0
32
3
15
32
42
3
8
18
35
14
16
10
8
12
29
47
24
4
7
39
21
51
66
44
57
64
58
51
85


In [63]:
import xarray as xr
import numpy as np

result = []
for i in range(1982, 2024):
    # 打开每年的数据文件
    data = xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt = data['z'] / 9.80665  # 将位势高度转换成米
    res_time = data['valid_time']
    
    # 找到与 common_times 相匹配的时间步
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    hgt500 = hgt.sel(valid_time=common_times)  # 每年的 hgt 数据
    hgtwp = hgt500.loc[:, 500, 60:10, 110:180]  # 选择区域
    mewp = hgtwp.mean('valid_time')  # 计算每年的平均位势高度
    
    # 使用 numpy 条件筛选来替代嵌套循环
    cond_5880_5890 = (mewp >= 5880) & (mewp < 5890)
    cond_5890_5900 = (mewp >= 5890) & (mewp < 5900)
    cond_5900_5910 = (mewp >= 5900) & (mewp < 5910)
    cond_5910_5920 = (mewp >= 5910) & (mewp < 5920)

    # 计算每个条件下的得分
    resum = np.sum(cond_5880_5890) * 1 + np.sum(cond_5890_5900) * 2 + np.sum(cond_5900_5910) * 3 + np.sum(cond_5910_5920) * 4

    print(f"Year: {i}, Score: {resum}")
    result.append(resum)


Year: 1982, Score: <xarray.DataArray 'z' ()> Size: 4B
array(2)
Coordinates:
    number          int64 8B 0
    pressure_level  float64 8B 500.0
Year: 1983, Score: <xarray.DataArray 'z' ()> Size: 4B
array(23)
Coordinates:
    number          int64 8B 0
    pressure_level  float64 8B 500.0
Year: 1984, Score: <xarray.DataArray 'z' ()> Size: 4B
array(0)
Coordinates:
    number          int64 8B 0
    pressure_level  float64 8B 500.0
Year: 1985, Score: <xarray.DataArray 'z' ()> Size: 4B
array(0)
Coordinates:
    number          int64 8B 0
    pressure_level  float64 8B 500.0
Year: 1986, Score: <xarray.DataArray 'z' ()> Size: 4B
array(2)
Coordinates:
    number          int64 8B 0
    pressure_level  float64 8B 500.0
Year: 1987, Score: <xarray.DataArray 'z' ()> Size: 4B
array(20)
Coordinates:
    number          int64 8B 0
    pressure_level  float64 8B 500.0
Year: 1988, Score: <xarray.DataArray 'z' ()> Size: 4B
array(14)
Coordinates:
    number          int64 8B 0
    pressure_level  float6

In [65]:
import xarray as xr
import numpy as np

result = []
for i in range(1982, 2024):
    # 打开每年的数据文件
    data = xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt = data['z'] / 9.80665  # 将位势高度转换成米
    res_time = data['valid_time']
    
    # 找到与 common_times 相匹配的时间步
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    hgt500 = hgt.sel(valid_time=common_times)  # 每年的 hgt 数据
    hgtwp = hgt500.loc[:, 500, 60:10, 110:180]  # 选择区域
    mewp = hgtwp.mean('valid_time')  # 计算每年的平均位势高度
    
    # 只筛选出位势高度大于或等于5880米的部分
    mask = mewp >= 5880
    
    # 计算每个格点的得分 (mewp - 5880) // 10, 超过5880米每增加10米，分数加1
    scores = np.floor((mewp - 5880) / 10).astype(int)
    
    # 使用 xr.where 根据布尔条件筛选，其他位置设为 NaN
    filtered_scores = xr.where(mask, scores, np.nan)
    
    # 计算符合条件的分数总和，忽略 NaN
    resum = np.nansum(filtered_scores)

    print(f"Year: {i}, Score: {resum}")
    result.append(resum)


Year: 1982, Score: 0.0
Year: 1983, Score: 4.0
Year: 1984, Score: 0.0
Year: 1985, Score: 0.0
Year: 1986, Score: 0.0
Year: 1987, Score: 1.0
Year: 1988, Score: 0.0
Year: 1989, Score: 0.0
Year: 1990, Score: 8.0
Year: 1991, Score: 12.0
Year: 1992, Score: 0.0
Year: 1993, Score: 6.0
Year: 1994, Score: 0.0
Year: 1995, Score: 11.0
Year: 1996, Score: 0.0
Year: 1997, Score: 0.0
Year: 1998, Score: 14.0
Year: 1999, Score: 21.0
Year: 2000, Score: 0.0
Year: 2001, Score: 0.0
Year: 2002, Score: 5.0
Year: 2003, Score: 14.0
Year: 2004, Score: 4.0
Year: 2005, Score: 3.0
Year: 2006, Score: 2.0
Year: 2007, Score: 0.0
Year: 2008, Score: 1.0
Year: 2009, Score: 9.0
Year: 2010, Score: 19.0
Year: 2011, Score: 7.0
Year: 2012, Score: 0.0
Year: 2013, Score: 1.0
Year: 2014, Score: 16.0
Year: 2015, Score: 3.0
Year: 2016, Score: 20.0
Year: 2017, Score: 22.0
Year: 2018, Score: 18.0
Year: 2019, Score: 22.0
Year: 2020, Score: 27.0
Year: 2021, Score: 23.0
Year: 2022, Score: 14.0
Year: 2023, Score: 34.0


# 强度指数

In [53]:
import xarray as xr
import numpy as np

result = []
for i in range(1982, 2024):
    # 打开每年的数据文件
    data = xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt = data['z'] / 9.80665  # 将位势高度转换成米
    res_time = data['valid_time']
    
    # 找到与 common_times 相匹配的时间步
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    hgt500 = hgt.sel(valid_time=common_times)  # 每年的 hgt 数据
    hgtwp = hgt500.loc[:, 500, 60:10, 110:180]  # 选择区域

    # 计算每个时间段和空间点的得分 (向量化处理，包括所有大于5880米的情况)
    scores = np.where(hgtwp >= 5880, np.floor((hgtwp - 5880) / 10).astype(int), 0)
    
    # 对每个时间段求总和，然后计算年平均得分
    time_average_score = np.mean(np.sum(scores, axis=(1, 2)))  # 在纬度和经度方向上求和后，对时间求平均

    result.append(time_average_score)

    print(f"Year: {i}, Average Score: {time_average_score}")


Year: 1982, Average Score: 22.20689655172414
Year: 1983, Average Score: 42.1544885177453
Year: 1984, Average Score: 10.533954727030626
Year: 1985, Average Score: 23.965753424657535
Year: 1986, Average Score: 29.765205091937766
Year: 1987, Average Score: 44.6817472698908
Year: 1988, Average Score: 36.48090277777778
Year: 1989, Average Score: 35.96128170894526
Year: 1990, Average Score: 42.57715133531157
Year: 1991, Average Score: 42.99869281045751
Year: 1992, Average Score: 23.88015978695073
Year: 1993, Average Score: 35.034108527131785
Year: 1994, Average Score: 25.787280701754387
Year: 1995, Average Score: 50.448
Year: 1996, Average Score: 30.472511144130756
Year: 1997, Average Score: 37.81188118811881
Year: 1998, Average Score: 58.21967213114754
Year: 1999, Average Score: 49.28699551569507
Year: 2000, Average Score: 22.353521126760562
Year: 2001, Average Score: 35.084151472650774
Year: 2002, Average Score: 42.83065953654189
Year: 2003, Average Score: 47.53125
Year: 2004, Average Scor

In [8]:
tomecur=[1.7939099453090457,
 1.5379027556117502,
 1.6764343172939071,
 1.4267147056509843,
 2.0711415742552224,
 1.3826291240693405,
 2.1402029145516352,
 1.3789691149420602,
 1.514161941795993,
 1.6437862980897897,
 1.7440951421065518,
 1.3653656171427437,
 1.756337608314105,
 1.7574158310933499,
 1.831228943487786,
 1.9389792366738476,
 1.5826751739081693,
 1.358554408436718,
 1.3804501703331484,
 1.4829805487553882,
 1.4962611808040724,
 1.6033931380661663,
 1.431700533283194,
 1.4119012781017781,
 1.5821405561616255,
 1.544820577001158,
 1.3347658844676835,
 1.4600357360792833,
 1.6311453240641929,
 1.749794636688014,
 1.5397778802464868,
 1.499573631780583,
 1.437733682587956,
 1.6213167048210742,
 1.5144458261696698,
 1.6824298870331225,
 1.3644764924813129,
 1.4234313127976026,
 1.431067629752025,
 1.3680763804795113,
 1.4903608646532545,
 1.538733687144302]

In [70]:
from scipy.stats import pearsonr
corr, p_value = pearsonr(result, tomecur)
corr, p_value

(0.051758826845985914, 0.744779329401059)

In [72]:
import xarray as xr
import numpy as np

# 地球半径 (单位：公里)
R = 6371

# 计算网格面积函数
def calculate_grid_area(lat, lon_resolution=5, lat_resolution=5):
    # 将纬度转换为弧度
    lat_radians = np.deg2rad(lat)
    # 计算每个纬度网格点的面积 (km^2)
    grid_area = (R**2 * np.deg2rad(lon_resolution) * np.deg2rad(lat_resolution) * np.cos(lat_radians))
    return grid_area

# 计算西太平洋副高面积指数
result = []
for i in range(1982, 2024):
    # 打开每年的数据文件
    data = xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt = data['z'] / 9.80665  # 将位势高度转换成米
    res_time = data['valid_time']
    
    # 找到与 common_times 相匹配的时间步
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    hgt500 = hgt.sel(valid_time=common_times)  # 每年的 hgt 数据
    hgtwp = hgt500.loc[:, 500, 60:10, 110:180]  # 选择区域

    # 筛选位势高度大于等于5880 gpm 的区域
    high_area = hgtwp.where(hgtwp >= 5880, drop=True)
    
    # 计算每个网格的面积
    latitudes = high_area['latitude'].values
    grid_areas = calculate_grid_area(latitudes)
    
    # 计算总面积 (只计算筛选出的区域)
    total_area = np.nansum(grid_areas)
    
    result.append(total_area)

#     print(f"Year: {i}, Western Pacific Subtropical High Area: {total_area} km^2")
from scipy.stats import pearsonr
corr, p_value = pearsonr(result, tomecur)
corr, p_value

(0.051758826845985914, 0.744779329401059)

In [73]:
import xarray as xr
import numpy as np

# 地球半径 (单位：公里)
R = 6371

# 计算网格面积函数
def calculate_grid_area(lat, lon_resolution=5, lat_resolution=5):
    # 将纬度转换为弧度
    lat_radians = np.deg2rad(lat)
    # 计算每个纬度网格点的面积 (km^2)
    grid_area = (R**2 * np.deg2rad(lon_resolution) * np.deg2rad(lat_resolution) * np.cos(lat_radians))
    return grid_area

# 计算西太平洋副高强度指数
result = []
for i in range(1982, 2024):
    # 打开每年的数据文件
    data = xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt = data['z'] / 9.80665  # 将位势高度转换成米
    res_time = data['valid_time']
    
    # 找到与 common_times 相匹配的时间步
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    hgt500 = hgt.sel(valid_time=common_times)  # 每年的 hgt 数据
    hgtwp = hgt500.loc[:, 500, 60:10, 110:180]  # 选择区域

    # 筛选位势高度大于等于5880 gpm 的区域
    high_area = hgtwp.where(hgtwp >= 5880, drop=True)

    # 计算位势高度差值 (位势高度 - 5870 gpm)
    height_diff = high_area - 5870

    # 计算每个网格的面积
    latitudes = high_area['latitude'].values
    grid_areas = calculate_grid_area(latitudes)

    # 计算每个格点的强度贡献 (位势高度差值 × 网格面积)
    intensity_contributions = height_diff * grid_areas[:, np.newaxis]

    # 计算强度指数：所有格点的强度贡献累积值
    total_intensity = np.nansum(intensity_contributions)

    result.append(total_intensity)

    print(f"Year: {i}, Western Pacific Subtropical High Intensity: {total_intensity}")
from scipy.stats import pearsonr
corr, p_value = pearsonr(result, tomecur)
corr, p_value

Year: 1982, Western Pacific Subtropical High Intensity: 101865788092.00775
Year: 1983, Western Pacific Subtropical High Intensity: 118340728518.31296
Year: 1984, Western Pacific Subtropical High Intensity: 56076564907.79847
Year: 1985, Western Pacific Subtropical High Intensity: 75988424266.5902
Year: 1986, Western Pacific Subtropical High Intensity: 118905941891.26874
Year: 1987, Western Pacific Subtropical High Intensity: 166025050673.81033
Year: 1988, Western Pacific Subtropical High Intensity: 119953111392.41504
Year: 1989, Western Pacific Subtropical High Intensity: 141611801553.0988
Year: 1990, Western Pacific Subtropical High Intensity: 153378973338.15793
Year: 1991, Western Pacific Subtropical High Intensity: 170387750186.0485
Year: 1992, Western Pacific Subtropical High Intensity: 108443029242.17996
Year: 1993, Western Pacific Subtropical High Intensity: 131476238051.87065
Year: 1994, Western Pacific Subtropical High Intensity: 136999545169.41356
Year: 1995, Western Pacific Su

(-0.1750862549588258, 0.26741340336231956)

In [78]:
import xarray as xr
import numpy as np

# 计算西太平洋副高西伸脊点指数
result = []
for i in range(1982, 2024):
    # 打开每年的数据文件
    data = xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt = data['z'] / 9.80665  # 将位势高度转换成米
    res_time = data['valid_time']
    
    # 找到与 common_times 相匹配的时间步
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    hgt500 = hgt.sel(valid_time=common_times)  # 每年的 hgt 数据
    hgtwp = hgt500.loc[:, 500, 60:10, 90:180]  # 选择区域

    # 使用 xr.apply_ufunc 应用 np.isclose
    contour5880 = xr.apply_ufunc(np.isclose, hgtwp, 5880, kwargs={"atol": 10})

    # 筛选出等值线对应的网格
    contour5880 = hgtwp.where(contour5880, drop=True)

    # 找到5880 gpm 等值线的最西端经度
    if contour5880.size > 0:
        # 获取所有5880 gpm 等值线的经度值
        westernmost_longitude = contour5880['longitude'].min().values
        result.append(westernmost_longitude)

        print(f"Year: {i}, Westernmost Ridge Point: {westernmost_longitude}°E")
    else:
        result.append(np.nan)
        print(f"Year: {i}, No 5880 gpm contour found")


Year: 1982, Westernmost Ridge Point: 90.0°E
Year: 1983, Westernmost Ridge Point: 90.0°E
Year: 1984, Westernmost Ridge Point: 90.0°E
Year: 1985, Westernmost Ridge Point: 90.0°E
Year: 1986, Westernmost Ridge Point: 90.0°E
Year: 1987, Westernmost Ridge Point: 90.0°E
Year: 1988, Westernmost Ridge Point: 90.0°E
Year: 1989, Westernmost Ridge Point: 90.0°E
Year: 1990, Westernmost Ridge Point: 90.0°E
Year: 1991, Westernmost Ridge Point: 90.0°E
Year: 1992, Westernmost Ridge Point: 90.0°E
Year: 1993, Westernmost Ridge Point: 90.0°E
Year: 1994, Westernmost Ridge Point: 90.0°E
Year: 1995, Westernmost Ridge Point: 90.0°E
Year: 1996, Westernmost Ridge Point: 90.0°E
Year: 1997, Westernmost Ridge Point: 90.0°E
Year: 1998, Westernmost Ridge Point: 90.0°E
Year: 1999, Westernmost Ridge Point: 90.0°E
Year: 2000, Westernmost Ridge Point: 90.0°E
Year: 2001, Westernmost Ridge Point: 90.0°E
Year: 2002, Westernmost Ridge Point: 90.0°E
Year: 2003, Westernmost Ridge Point: 90.0°E
Year: 2004, Westernmost Ridge Po

In [10]:
import xarray as xr
import numpy as np

# 创建一个空的数组用于存储每年的西伸脊点指数
wpsh_wp = np.zeros((2024 - 1982))  # 保存1982到2023年每年的结果

for i in range(1982, 2024):
    # 打开每年的数据文件
    data = xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt = data['z'] / 9.80665  # 将位势高度转换成米
    res_time = data['valid_time']
    
    # 找到与 common_times 相匹配的时间步
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    
    if len(common_times) > 0:
        hgt500 = hgt.sel(valid_time=common_times)  # 每年的 hgt 数据
        hgtwp = hgt500.sel(latitude=slice(60, 10), longitude=slice(90, 180))  # 选择区域
        
        # 对夏季6、7、8月的数据进行分析
        z1 = hgtwp.loc[hgtwp['valid_time'].dt.month.isin([6, 7, 8,9])]
        lon = z1.longitude.values
        z1_array = z1.mean(axis=1)
        
        # 找到每年的西伸脊点
        for j in range(z1_array.shape[2]):  # 对每个经度进行循环
            if z1_array[:, :, j].max() >= 5880:  # 查找5880gpm等位线
                wpsh_wp[i - 1982] = lon[j]
                break
        
        # 如果没有5880gpm等位线，再找5870gpm等位线
        if wpsh_wp[i - 1982] == 0:
            for j in range(z1_array.shape[2]):
                if z1_array[:, :, j].max() >= 5870:
                    wpsh_wp[i - 1982] = lon[j]
                    break

# 输出结果
wpsh_wp


array([90., 90., 90., 90., 90., 90., 90., 90., 90., 90., 90., 90., 90.,
       90., 90., 90., 90., 90., 90., 90., 90., 90., 90., 90., 90., 90.,
       90., 90., 90., 90., 90., 90., 90., 90., 90., 90., 90., 90., 90.,
       90., 90., 90.])

In [9]:
import xarray as xr
import numpy as np

#面积指数
a=[]
for i in range(1982, 2024):
    # 打开每年的数据文件
    data = xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt = data['z'] / 9.80665  # 将位势高度转换成米
    res_time = data['valid_time']
    
      # 找到与 common_times 相匹配的时间步
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    hgt500 = hgt.sel(valid_time=common_times)  # 每年的 hgt 数据
    hgtwp = hgt500.loc[:, 500, 60:10, 90:180]  # 选择区域
    z2_array = hgtwp.mean('valid_time')
    wpsh_area = np.sum(np.array(z2_array)>5880)
#     print(wpsh_area)
    a.append(wpsh_area)
from scipy.stats import pearsonr
corr, p_value = pearsonr(a, tomecur)
corr, p_value

(-0.2489591420009337, 0.11185828371752002)

In [10]:
import xarray as xr
import numpy as np

#面积指数
b=[]
for i in range(1982, 2024):
    # 打开每年的数据文件
    data = xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt = data['z'] / 9.80665  # 将位势高度转换成米
    res_time = data['valid_time']
    
      # 找到与 common_times 相匹配的时间步
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    hgt500 = hgt.sel(valid_time=common_times)  # 每年的 hgt 数据
    hgtwp = hgt500.loc[:, 500, 60:10, 110:180]  # 选择区域
    z3_array =np.array( hgtwp.mean('valid_time'))
    z3_array = z3_array-5880
    z3_array[z3_array<0]=0
    wpsh_int = np.sum(z3_array)/10
    b.append(wpsh_int)
    
from scipy.stats import pearsonr
corr, p_value = pearsonr(b, tomecur)
corr, p_value

(-0.27732979355938536, 0.07538349271583084)

In [11]:
import xarray as xr
import numpy as np
wpsh_wp = np.zeros((42))
#面积指数

for i in range(1982, 2024):
    # 打开每年的数据文件
    data = xr.open_dataset(f'E:/data/daily5/geopotential/geo_{i}.nc')
    hgt = data['z'] / 9.80665  # 将位势高度转换成米
    res_time = data['valid_time']
    
      # 找到与 common_times 相匹配的时间步
    common_times = np.intersect1d(timec1_datetime, res_time.values)
    hgt500 = hgt.sel(valid_time=common_times)  # 每年的 hgt 数据
    hgtwp = hgt500.loc[:, 500, 60:10, 110:180]  # 选择区域
    lon = hgtwp['longitude']
    z1_array =np.array( hgtwp.mean('valid_time'))
   
    
#     wpsh_wp = np.zeros((42))

    for j in range(z1_array.shape[1]):
        if z1_array[:,j].max()>=5880:
            wpsh_wp[i-1982] = lon[j]
            break
    if wpsh_wp[i-1982]==0:
        for j in range(z1_array.shape[1]):
            if z1_array[:,j].max()>=5870:
                wpsh_wp[i-1982] = lon[j]
                break
wpsh_wp
from scipy.stats import pearsonr
corr, p_value = pearsonr(wpsh_wp, tomecur)
corr, p_value

(0.343150423713988, 0.026095169743994555)

In [12]:
wpsh_wp

array([155., 140., 160., 140., 165., 145., 155., 155., 150., 145., 170.,
       140., 150., 145., 155., 145., 140., 140., 150., 160., 150., 145.,
       140., 140., 155., 155., 155., 145., 135., 150., 150., 145., 140.,
       150., 140., 120., 140., 135., 135., 140., 110., 115.])